# FreshBasket AI - Phase 3: Exploratory Data Analysis (EDA)

**Objective:** Uncover demand patterns, seasonality, promotional elasticity, store characteristics, and category dynamics to inform 13-week FMCG demand forecasting.

### Dataset Overview
- **Source:** `data/processed/cleaned_sales.csv`
- **Scope:** 54 Stores across Ecuador, 33 FMCG Product Families
- **Timeline:** 2013-01-01 to 2017-08-15 (1,688 continuous calendar days, 3,008,016 rows)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Set project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_sales.csv'

print(f"Loading dataset from: {PROCESSED_DATA_PATH}")

## 1. Load Cleaned Dataset

In [ ]:
dtypes = {
    'id': np.int64,
    'store_nbr': np.int16,
    'family': 'category',
    'sales': np.float64,
    'onpromotion': np.int32,
    'city': 'category',
    'state': 'category',
    'type': 'category',
    'cluster': np.int8,
    'dcoilwtico': np.float32,
    'is_holiday': bool,
    'is_national_holiday': bool,
    'is_regional_holiday': bool,
    'is_local_holiday': bool,
    'is_event': bool,
    'is_work_day': bool,
    'transactions': np.int32,
    'is_imputed_closure': bool,
    'is_store_active': bool,
    'is_store_open': bool,
}
df = pd.read_csv(PROCESSED_DATA_PATH, dtype=dtypes)
df['date'] = pd.to_datetime(df['date'])
print(f"Total Rows: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
df.head()

## 2. Overall Sales Velocity & Long-Term Trend

In [ ]:
daily_sales = df.groupby('date')['sales'].sum().reset_index()
daily_sales['sales_rolling_30d'] = daily_sales['sales'].rolling(30, center=True).mean()

fig = px.line(
    daily_sales,
    x='date',
    y='sales',
    title='Total Daily Sales Across All Stores (2013-2017)',
    labels={'sales': 'Total Sales (Units)', 'date': 'Date'}
)
fig.add_scatter(x=daily_sales['date'], y=daily_sales['sales_rolling_30d'], name='30-Day Moving Average', line=dict(color='red', width=2))
fig.show()

## 3. Product Family Breakdown (Pareto Analysis)

In [ ]:
family_summary = df.groupby('family', observed=False)['sales'].sum().reset_index()
family_summary = family_summary.sort_values('sales', ascending=False)
family_summary['pct_share'] = (family_summary['sales'] / family_summary['sales'].sum()) * 100
family_summary['cum_share'] = family_summary['pct_share'].cumsum()

fig = px.bar(
    family_summary.head(15),
    x='sales',
    y='family',
    orientation='h',
    text='pct_share',
    title='Top 15 Product Families by Total Sales Volume and Share',
    labels={'sales': 'Total Units Sold', 'family': 'Product Family'}
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 4. Promotional Uplift Analysis

In [ ]:
open_stores = df[df['is_store_open']]
promo_analysis = open_stores.groupby('family', observed=False).apply(
    lambda g: pd.Series({
        'avg_no_promo': g.loc[g['onpromotion'] == 0, 'sales'].mean(),
        'avg_with_promo': g.loc[g['onpromotion'] > 0, 'sales'].mean(),
    })
).reset_index()
promo_analysis['uplift_pct'] = ((promo_analysis['avg_with_promo'] - promo_analysis['avg_no_promo']) / promo_analysis['avg_no_promo']) * 100
promo_analysis = promo_analysis.sort_values('uplift_pct', ascending=False).dropna()
print("Top 10 Categories by Promotional Uplift:")
promo_analysis.head(10)

## 5. Store Performance & Cluster Hierarchy

In [ ]:
store_summary = df.groupby(['store_nbr', 'type', 'cluster'], observed=False)['sales'].sum().reset_index()
fig = px.box(
    store_summary,
    x='type',
    y='sales',
    color='type',
    title='Store Total Sales Distribution by Store Format Type',
    labels={'sales': 'Cumulative Sales Units', 'type': 'Store Format Type'}
)
fig.show()

## 6. Holiday & Day-of-Week Seasonality

In [ ]:
open_stores['day_of_week'] = open_stores['date'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_sales = open_stores.groupby('day_of_week')['sales'].mean().reindex(dow_order).reset_index()

fig = px.bar(
    dow_sales,
    x='day_of_week',
    y='sales',
    title='Average Daily Sales by Day of Week (Weekend Demand Peak)',
    labels={'sales': 'Mean Daily Sales per Series', 'day_of_week': 'Day of Week'},
    color='sales',
    color_continuous_scale='Viridis'
)
fig.show()

## 7. Conclusions & Next Steps for Feature Engineering
- **Day-of-Week Encoding:** Crucial cyclical pattern with weekend surges (Saturday/Sunday peak).
- **13-Week Horizon Features:** Lags must be $\ge 14$ days to prevent test-period data leakage.
- **Promotional Features:** Rolling promotional intensity is a strong uplift predictor for impulse categories.
- **Store Segmentation:** Hierarchical grouping by Store Type and Cluster provides essential regularization.